# jupyter-loopback end-to-end demo

Everything is inline in this notebook. Read each cell to see exactly what integrating `jupyter-loopback` takes. Three moving parts:

1. **A jupyter-server extension** (the `loopback_demo._jupyter` package on disk) mounts the proxy.
2. **An in-kernel HTTP+WS server** (defined right below) serves some content.
3. **`autodetect_prefix`** tells the kernel which browser-reachable URL to hand out.

If the red square renders and the WebSocket echo answers back, the full round-trip works.

## 1. What the kernel sees

`jupyter-loopback` reads four environment variables. `JPY_SESSION_NAME` (or `JPY_PARENT_PID`) proves we're in a kernel; `JUPYTERHUB_SERVICE_PREFIX` or `JPY_BASE_URL` carry any per-user URL prefix.

In [ ]:
import os

for var in ("JPY_SESSION_NAME", "JPY_PARENT_PID", "JUPYTERHUB_SERVICE_PREFIX", "JPY_BASE_URL"):
    print(f"{var} = {os.environ.get(var)!r}")

## 2. Extension check

The `loopback_demo` package installs a one-line jupyter-server extension that calls `setup_proxy_handler(web_app, namespace='loopback-demo')`. When the Docker image boots JupyterLab, the extension auto-loads because the package ships `jupyter-config/jupyter_server_config.d/loopback-demo.json`.

In [ ]:
import inspect

from loopback_demo import _jupyter

print(inspect.getsource(_jupyter))

## 3. In-kernel HTTP + WebSocket server

A small Tornado app. Three routes:

- `GET /hello` returns JSON.
- `GET /image.png` returns a 1×1 red PNG. The HTML display below upscales it so it's visible.
- `WS  /ws` echoes messages back.

Handlers set `Access-Control-Allow-Origin: *` so the browser can embed responses in iframes (e.g. if you ever render this through a folium HTML output).

In [ ]:
import asyncio
import json
import threading
import time

from tornado.httpserver import HTTPServer
from tornado.ioloop import IOLoop
from tornado.testing import bind_unused_port
from tornado.web import Application, RequestHandler
from tornado.websocket import WebSocketHandler

# Minimal 1x1 red PNG, hand-rolled so the demo has no Pillow dep.
RED_PNG = (
    b"\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x00\x01\x00\x00\x00"
    b"\x01\x08\x02\x00\x00\x00\x90wS\xde\x00\x00\x00\x0cIDATx\x9cc\xf8"
    b"\xcf\xc0\x00\x00\x00\x03\x00\x01\x84\xd0\xf9]\x00\x00\x00\x00IEND"
    b"\xaeB`\x82"
)


class Hello(RequestHandler):
    def get(self):
        self.set_header("Access-Control-Allow-Origin", "*")
        self.set_header("Content-Type", "application/json")
        self.write(json.dumps({"message": "hello from the kernel", "now": time.time()}))


class Image(RequestHandler):
    def get(self):
        self.set_header("Access-Control-Allow-Origin", "*")
        self.set_header("Content-Type", "image/png")
        self.write(RED_PNG)


class EchoWS(WebSocketHandler):
    def check_origin(self, origin):
        return True

    def open(self):
        self.count = 0

    async def on_message(self, message):
        self.count += 1
        await self.write_message(f"[{self.count}] {message}")


# Bind a loopback port up front so we know the number before starting.
sock, port = bind_unused_port(address="127.0.0.1")
print(f"server will bind to 127.0.0.1:{port}")


def run():
    asyncio.set_event_loop(asyncio.new_event_loop())
    loop = IOLoop.current()
    app = Application(
        [
            (r"/hello", Hello),
            (r"/image.png", Image),
            (r"/ws", EchoWS),
        ]
    )
    server = HTTPServer(app)
    server.add_sockets([sock])
    ready.set()
    loops.append(loop)
    loop.start()


ready = threading.Event()
loops = []  # one-element list so run() can publish the loop back to us
thread = threading.Thread(target=run, name="demo-server", daemon=True)
thread.start()
ready.wait(timeout=5.0)
print("server running")

## 4. Build a browser-reachable URL

`autodetect_prefix('loopback-demo')` returns a root-absolute path template. `None` outside a Jupyter kernel so the same code can run locally against `127.0.0.1:<port>` directly.

In [ ]:
from jupyter_loopback import autodetect_prefix

prefix = autodetect_prefix("loopback-demo")
print("prefix template:", prefix)

http_url = f"{prefix.format(port=port)}" if prefix else f"http://127.0.0.1:{port}"
ws_url = f"{http_url}/ws"
print("http_url:", http_url)
print("ws_url:  ", ws_url)

## 5. Sanity check: kernel hits the loopback socket

The kernel and the server share a process, so this goes straight through `127.0.0.1`. If this fails, the server is broken and the proxy tests below will too.

In [ ]:
import requests

r = requests.get(f"http://127.0.0.1:{port}/hello", timeout=5)
print("status:", r.status_code)
print("body:  ", r.json())

## 6. Browser round-trip through the proxy

The HTML below uses `http_url` / `ws_url` from step 4. The browser hits the Jupyter server, the extension routes to `127.0.0.1:<port>`. Three checks:

- **JSON link**: opens `/hello` in a new tab.
- **Inline image**: 1×1 red PNG, CSS-upscaled to 80×80 with `image-rendering: pixelated`. If you see a red square, binary bodies survive proxying.
- **WebSocket echo**: type a message, press Enter. Server echoes back prefixed with a count.

In [ ]:
from IPython.display import HTML

html = f'''
<div style="border:1px solid #ddd;border-radius:6px;padding:12px;
            font-family:system-ui,sans-serif;max-width:640px">
  <p style="margin:0 0 8px">Bound to <code>127.0.0.1:{port}</code> in the kernel.</p>
  <ul style="margin:0 0 8px">
    <li><a href="{http_url}/hello" target="_blank"><code>GET {http_url}/hello</code></a></li>
    <li><code>GET {http_url}/image.png</code>:
      <img src="{http_url}/image.png" alt="red square"
           style="vertical-align:middle;width:80px;height:80px;
                  image-rendering:pixelated;border:1px solid #ccc;
                  border-radius:4px;margin-left:6px"/></li>
    <li><code>WS  {ws_url}</code></li>
  </ul>
  <input id="lpbk-in-{port}" type="text" placeholder="type, press Enter"
         style="width:100%;padding:6px;border:1px solid #ccc;border-radius:4px"/>
  <pre id="lpbk-log-{port}"
       style="margin-top:8px;background:#f7f7f7;padding:8px;border-radius:4px;
              max-height:140px;overflow:auto"></pre>
  <script>
    (function() {{
      var wsPath = {json.dumps(ws_url)};
      var abs = new URL(wsPath, window.location.origin);
      abs.protocol = window.location.protocol === "https:" ? "wss:" : "ws:";
      var input = document.getElementById("lpbk-in-{port}");
      var log = document.getElementById("lpbk-log-{port}");
      var ws = new WebSocket(abs.toString());
      ws.onopen = function()  {{ log.textContent += "[open " + abs + "]\\n"; }};
      ws.onclose = function() {{ log.textContent += "[close]\\n"; }};
      ws.onerror = function() {{ log.textContent += "[error]\\n"; }};
      ws.onmessage = function(ev) {{
        log.textContent += "< " + ev.data + "\\n";
        log.scrollTop = log.scrollHeight;
      }};
      input.addEventListener("keydown", function(e) {{
        if (e.key !== "Enter") return;
        var v = input.value; input.value = "";
        log.textContent += "> " + v + "\\n";
        log.scrollTop = log.scrollHeight;
        ws.send(v);
      }});
    }})();
  </script>
</div>
'''

HTML(html)

## 7. Cleanup

Stop the background IOLoop. In a real library you'd do this in a shutdown hook.

In [ ]:
loops[0].add_callback(loops[0].stop)
thread.join(timeout=5)
print("stopped")